# 01 — Audit data dan persiapan wajah

Salinan publik dari alur eksperimen. Tidak memuat dataset, nama peserta, embedding, foto, output cell, atau path komputer penulis. Gunakan dataset sendiri yang berhak diproses. Hasil numerik akan bergantung pada dataset dan tidak harus sama dengan README.

Setelah dijalankan pada data pribadi, output notebook dapat kembali mengandung nama atau wajah. Jalankan script pembersih sebelum commit. Lihat `notebooks_public/README.md` untuk prasyarat dan batas reproduksi.

## 1. Lokasi data

Gunakan `dataset/raw/<label_orang>/<foto>`. Sebaiknya gunakan label pseudonim. Dataset harus berupa gambar yang dapat didekode. Protokol notebook selanjutnya membutuhkan 3–4 foto berbeda per identitas.

In [ ]:
from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
RAW_DIR = PROJECT_ROOT / "dataset" / "raw"
MODEL_ROOT = PROJECT_ROOT / "weights" / "insightface"
SHOW_PRIVATE_PREVIEWS = False
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Dataset milik sendiri belum tersedia: {RAW_DIR}")

## 2. Audit keterbacaan

In [ ]:
import pandas as pd
from PIL import Image

person_dirs = sorted(
    [p for p in RAW_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.casefold()
)

# File sistem Windows yang bukan bagian dari dataset.
ignored_names = {"thumbs.db", "desktop.ini", ".ds_store"}

rows = []

for person_dir in person_dirs:
    files = sorted(
        [
            p for p in person_dir.rglob("*")
            if p.is_file()
            and p.name.casefold() not in ignored_names
        ],
        key=lambda p: str(p).casefold()
    )

    for file_path in files:
        row = {
            "person_name": person_dir.name,
            "filename": file_path.name,
            "relative_path": str(file_path.relative_to(RAW_DIR)),
            "extension": file_path.suffix.lower(),
            "actual_format": None,
            "width": None,
            "height": None,
            "color_mode": None,
            "readable": False,
            "error": None,
        }

        try:
            # Memeriksa struktur file gambar.
            with Image.open(file_path) as img:
                row["actual_format"] = img.format
                img.verify()

            # Buka ulang dan baca seluruh piksel.
            # File yang bisa dibuka belum tentu bisa didekode penuh.
            with Image.open(file_path) as img:
                img.load()

                row["width"], row["height"] = img.size
                row["color_mode"] = img.mode

                # Memastikan gambar bisa dipakai sebagai RGB.
                rgb = img.convert("RGB")
                rgb.load()

            row["readable"] = True

        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"

        rows.append(row)

columns = [
    "person_name", "filename", "relative_path", "extension",
    "actual_format", "width", "height", "color_mode",
    "readable", "error"
]

audit_df = pd.DataFrame(rows, columns=columns)

print("Jumlah folder orang :", len(person_dirs))
print("Jumlah file diperiksa:", len(audit_df))

display(audit_df.head())
if audit_df.empty:
    raise ValueError("Tidak ada file gambar untuk diproses.")

## 3. Ringkasan jumlah foto

In [ ]:
EXPECTED_PEOPLE = len(person_dirs)
EXPECTED_PHOTOS = 4

summary_df = (
    audit_df
    .groupby("person_name")
    .agg(
        jumlah_file=("relative_path", "size"),
        gambar_terbaca=("readable", "sum")
    )
    .reindex(
        [p.name for p in person_dirs],
        fill_value=0
    )
    .rename_axis("person_name")
    .reset_index()
)

summary_df["gambar_terbaca"] = (
    summary_df["gambar_terbaca"].astype(int)
)

summary_df["gagal_dibaca"] = (
    summary_df["jumlah_file"] - summary_df["gambar_terbaca"]
)

summary_df["sesuai_4_foto"] = (
    (summary_df["jumlah_file"] == EXPECTED_PHOTOS)
    & (summary_df["gambar_terbaca"] == EXPECTED_PHOTOS)
)

print(f"Folder orang: {len(person_dirs)} / {EXPECTED_PEOPLE}")
print(f"Total file  : {len(audit_df)}")
print(f"Terbaca     : {int(audit_df['readable'].sum())}")
print(f"Gagal dibaca: {int((~audit_df['readable']).sum())}")

display(summary_df)

## 4. Detail audit

In [ ]:
print("Folder yang jumlah atau keterbacaan fotonya belum sesuai:")
display(
    summary_df.loc[~summary_df["sesuai_4_foto"]]
)

print("Detail file yang gagal dibaca:")
display(
    audit_df.loc[
        ~audit_df["readable"],
        ["person_name", "filename", "extension", "error"]
    ]
)

print("Distribusi format gambar yang berhasil dibaca:")
display(
    audit_df.loc[audit_df["readable"], "actual_format"]
    .value_counts()
    .rename_axis("format")
    .reset_index(name="jumlah")
)

## 5. Hash piksel dan orientasi

In [ ]:
import hashlib
from PIL import Image, ImageOps

detail_rows = []

for record in audit_df.loc[audit_df["readable"]].to_dict("records"):
    file_path = RAW_DIR / record["relative_path"]

    result = {
        **record,
        "n_frames": None,
        "oriented_width": None,
        "oriented_height": None,
        "pixel_hash": None,
        "detail_error": None,
    }

    try:
        with Image.open(file_path) as img:
            result["n_frames"] = getattr(img, "n_frames", 1)

            # Audit gambar utama; tidak mengubah file asli.
            img.seek(0)

            rgb = ImageOps.exif_transpose(img).convert("RGB")
            rgb.load()

            result["oriented_width"] = rgb.width
            result["oriented_height"] = rgb.height

            hasher = hashlib.sha256()
            hasher.update(
                f"RGB:{rgb.width}:{rgb.height}:".encode("utf-8")
            )
            hasher.update(rgb.tobytes())

            result["pixel_hash"] = hasher.hexdigest()

    except Exception as exc:
        result["detail_error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    detail_rows.append(result)

audit_detail_df = pd.DataFrame(detail_rows)

print("File diperiksa:", len(audit_detail_df))
print(
    "Gagal pada pemeriksaan lanjutan:",
    audit_detail_df["detail_error"].notna().sum()
)

## 6. Duplikasi identik

In [ ]:
valid_hash_df = audit_detail_df.dropna(subset=["pixel_hash"])

duplicate_df = (
    valid_hash_df.loc[
        valid_hash_df["pixel_hash"].duplicated(keep=False)
    ]
    .sort_values(["pixel_hash", "person_name"])
    .copy()
)

print("Jumlah kelompok duplikat:",
      duplicate_df["pixel_hash"].nunique())
print("Jumlah file dalam kelompok duplikat:",
      len(duplicate_df))

display(
    duplicate_df[
        ["person_name", "filename", "relative_path", "pixel_hash"]
    ]
)

## 7. Frame dan ukuran gambar

In [ ]:
multiframe_df = audit_detail_df.loc[
    audit_detail_df["n_frames"].fillna(0) > 1
]

print("File dengan beberapa frame:", len(multiframe_df))

display(
    multiframe_df[
        ["person_name", "filename", "actual_format", "n_frames"]
    ]
)

print("Ringkasan ukuran gambar setelah koreksi orientasi:")

display(
    audit_detail_df[
        ["oriented_width", "oriented_height"]
    ].describe().round(1)
)

print("Error pemeriksaan lanjutan:")

display(
    audit_detail_df.loc[
        audit_detail_df["detail_error"].notna(),
        ["person_name", "filename", "detail_error"]
    ]
)

## 8. Pemeriksaan frame MPO

In [ ]:
import math
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

mpo_records = audit_detail_df.loc[
    audit_detail_df["actual_format"].eq("MPO")
].to_dict("records")

frame_results = []

CELL_WIDTH = 300
IMAGE_HEIGHT = 220
LABEL_HEIGHT = 30
COLUMNS = 3

for record in mpo_records:
    file_path = RAW_DIR / record["relative_path"]

    print("Orang :", record["person_name"])
    print("File  :", record["filename"])

    with Image.open(file_path) as source:
        frame_count = getattr(source, "n_frames", 1)
        grid_rows = math.ceil(frame_count / COLUMNS)

        sheet = Image.new(
            "RGB",
            (
                COLUMNS * CELL_WIDTH,
                grid_rows * (IMAGE_HEIGHT + LABEL_HEIGHT)
            ),
            "white"
        )

        draw = ImageDraw.Draw(sheet)

        for frame_index in range(frame_count):
            x = (frame_index % COLUMNS) * CELL_WIDTH
            y = (frame_index // COLUMNS) * (
                IMAGE_HEIGHT + LABEL_HEIGHT
            )

            result = {
                "person_name": record["person_name"],
                "relative_path": record["relative_path"],
                "frame_index": frame_index,
                "readable": False,
                "width": None,
                "height": None,
                "error": None,
            }

            try:
                source.seek(frame_index)
                source.load()

                frame = ImageOps.exif_transpose(source).convert("RGB")
                frame.load()

                result["width"], result["height"] = frame.size
                result["readable"] = True

                # Ukuran hanya diperkecil untuk pratinjau.
                frame.thumbnail((CELL_WIDTH, IMAGE_HEIGHT))

                paste_x = x + (CELL_WIDTH - frame.width) // 2
                paste_y = y + (IMAGE_HEIGHT - frame.height) // 2

                sheet.paste(frame, (paste_x, paste_y))

                label = (
                    f"Frame {frame_index} | "
                    f"{result['width']} x {result['height']}"
                )

            except Exception as exc:
                result["error"] = f"{type(exc).__name__}: {exc}"
                label = f"Frame {frame_index} | GAGAL DIBACA"

            draw.text(
                (x + 8, y + IMAGE_HEIGHT + 5),
                label,
                fill="black"
            )

            frame_results.append(result)

    if SHOW_PRIVATE_PREVIEWS:
        display(sheet)
    print()

mpo_frame_audit_df = pd.DataFrame(frame_results, columns=[
    "person_name", "relative_path", "frame_index", "readable",
    "width", "height", "error",
])

## 9. Ringkasan MPO

In [ ]:
print("Jumlah file MPO    :", len(mpo_records))
print("Total frame        :", len(mpo_frame_audit_df))
print("Frame terbaca      :", int(mpo_frame_audit_df["readable"].sum()))
print("Frame gagal dibaca :", int((~mpo_frame_audit_df["readable"]).sum()))

display(
    mpo_frame_audit_df.loc[
        ~mpo_frame_audit_df["readable"],
        ["person_name", "relative_path", "frame_index", "error"]
    ]
)

## 10. ID identitas

In [ ]:
if not audit_detail_df["readable"].all() or audit_detail_df["detail_error"].notna().any():
    raise ValueError("Perbaiki file yang gagal dibaca sebelum standardisasi.")

STANDARDIZED_DIR = PROJECT_ROOT / "dataset" / "standardized"
METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"

STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

identities_path = METADATA_DIR / "identities.csv"

current_names = sorted(
    audit_detail_df["person_name"].unique(),
    key=str.casefold
)

if identities_path.exists():
    # Gunakan pemetaan sebelumnya agar ID tidak berubah.
    identities_df = pd.read_csv(
        identities_path,
        dtype=str,
        keep_default_na=False
    )
else:
    identities_df = pd.DataFrame(
        columns=["person_id", "person_name"]
    )

if (
    identities_df["person_id"].duplicated().any()
    or identities_df["person_name"].duplicated().any()
):
    raise ValueError("Pemetaan identitas memiliki duplikat.")

known_names = set(identities_df["person_name"])

last_number = max(
    (
        int(person_id.split("_")[-1])
        for person_id in identities_df["person_id"]
    ),
    default=0
)

new_identities = []

for name in current_names:
    if name not in known_names:
        last_number += 1

        new_identities.append({
            "person_id": f"person_{last_number:03d}",
            "person_name": name,
        })

if new_identities:
    identities_df = pd.concat(
        [identities_df, pd.DataFrame(new_identities)],
        ignore_index=True
    )

identities_df.to_csv(
    identities_path,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah identitas:", len(identities_df))
display(identities_df.head())

## 11. Standardisasi RGB PNG

In [ ]:
import hashlib
from PIL import Image, ImageOps

person_id_map = dict(
    zip(
        identities_df["person_name"],
        identities_df["person_id"]
    )
)

conversion_rows = []
assigned_ids = set()

for record in audit_detail_df.to_dict("records"):
    person_id = person_id_map[record["person_name"]]
    original_relative_path = record["relative_path"]

    # ID stabil selama lokasi relatif file asli tetap sama.
    path_code = hashlib.sha256(
        original_relative_path.encode("utf-8")
    ).hexdigest()[:12]

    image_id = f"{person_id}_img_{path_code}"

    if image_id in assigned_ids:
        raise ValueError(f"ID gambar bertabrakan: {image_id}")

    assigned_ids.add(image_id)

    source_path = RAW_DIR / original_relative_path
    output_dir = STANDARDIZED_DIR / person_id
    output_path = output_dir / f"{image_id}.png"

    result = {
        "image_id": image_id,
        "person_id": person_id,
        "person_name": record["person_name"],
        "original_path": original_relative_path,
        "original_format": record["actual_format"],
        "selected_frame": 0,
        "standardized_path": str(
            output_path.relative_to(PROJECT_ROOT)
        ),
        "width": None,
        "height": None,
        "status": "failed",
        "error": None,
    }

    try:
        output_dir.mkdir(parents=True, exist_ok=True)

        with Image.open(source_path) as source:
            source.seek(0)
            source.load()

            rgb = ImageOps.exif_transpose(source).convert("RGB")
            rgb.load()

            expected_size = rgb.size
            rgb.save(output_path, format="PNG")

        # Pastikan hasil penyimpanan dapat dibaca kembali.
        with Image.open(output_path) as saved:
            saved.load()

            if saved.format != "PNG":
                raise ValueError("Format hasil bukan PNG.")

            if saved.mode != "RGB":
                raise ValueError("Mode hasil bukan RGB.")

            if saved.size != expected_size:
                raise ValueError("Ukuran gambar berubah.")

            result["width"], result["height"] = saved.size

        result["status"] = "success"

    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"

    conversion_rows.append(result)

standardization_df = pd.DataFrame(conversion_rows)

manifest_path = METADATA_DIR / "standardization_manifest.csv"

standardization_df.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8-sig"
)

print("Manifest tersimpan:", manifest_path)

## 12. Verifikasi standardisasi

In [ ]:
successful_df = standardization_df.loc[
    standardization_df["status"].eq("success")
]

failed_df = standardization_df.loc[
    standardization_df["status"].eq("failed")
]

print("Total file diproses :", len(standardization_df))
print("Berhasil            :", len(successful_df))
print("Gagal               :", len(failed_df))
print("Identitas berhasil  :", successful_df["person_id"].nunique())

print("\nDetail kegagalan:")
display(
    failed_df[
        ["person_name", "original_path", "error"]
    ]
)

print("Jumlah foto hasil per orang:")
standardized_summary_df = (
    successful_df
    .groupby(["person_id", "person_name"])
    .size()
    .reset_index(name="jumlah_foto")
)

display(standardized_summary_df)
if len(failed_df):
    raise ValueError("Perbaiki kegagalan standardisasi sebelum melanjutkan.")

## 13. Siapkan model lokal

Helper proyek mengunduh dan memverifikasi model bila diperlukan. Unduhan pertama memerlukan internet. Perhatikan ketentuan model untuk penelitian nonkomersial.

In [ ]:
from src.model_files import ensure_models

ensure_models(PROJECT_ROOT, {
    "detection_model": "weights/insightface/models/buffalo_l/det_10g.onnx",
    "recognition_model": "weights/insightface/models/buffalo_l/w600k_r50.onnx",
})

In [ ]:
import time
from insightface.app import FaceAnalysis

started = time.perf_counter()

print("Memuat model lokal...", flush=True)

face_detector = FaceAnalysis(
    name="buffalo_l",
    root=str(MODEL_ROOT),
    allowed_modules=["detection"],
    providers=["CPUExecutionProvider"]
)

print("Menyiapkan detector CPU...", flush=True)

face_detector.prepare(
    ctx_id=-1,
    det_size=(640, 640),
    det_thresh=0.5
)

print(
    f"Detector siap digunakan. "
    f"Waktu: {time.perf_counter() - started:.1f} detik",
    flush=True
)

## 14. Deteksi dengan checkpoint

Foto yang sudah tercatat tidak dideteksi ulang. Jika foto sumber atau konfigurasi detektor diubah, gunakan workspace eksperimen baru atau kelola ulang artefaknya secara sadar.

In [ ]:
AUDIT_PATH = METADATA_DIR / "face_detection_audit.csv"
SELECTION_PATH = METADATA_DIR / "manual_face_selection.csv"
import json
import time
import numpy as np

from PIL import Image
from tqdm.auto import tqdm

# Selalu baca progres terbaru dari disk.
standard_df = pd.read_csv(
    METADATA_DIR / "standardization_manifest.csv"
)

standard_df = standard_df.loc[
    standard_df["status"].eq("success")
].copy()

if not AUDIT_PATH.is_file():
    pd.DataFrame(columns=["image_id"]).to_csv(AUDIT_PATH, index=False)
audit_live_df = pd.read_csv(AUDIT_PATH)

if audit_live_df["image_id"].duplicated().any():
    raise ValueError("Ada image_id duplikat dalam CSV deteksi.")

pending_detection_df = standard_df.loc[
    ~standard_df["image_id"].isin(audit_live_df["image_id"])
].copy()

print("Sudah tercatat:", len(audit_live_df))
print("Akan diproses :", len(pending_detection_df))

records = pending_detection_df.to_dict("records")

for record in tqdm(
    records,
    desc="Melengkapi deteksi",
    unit="foto"
):
    started = time.perf_counter()

    result = {
        **record,
        "n_faces": None,
        "face_status": "error",
        "faces_json": "[]",
        "detector": "buffalo_l/det_10g",
        "det_thresh": 0.5,
        "det_size": "640x640",
        "detection_error": None,
        "detection_seconds": None,
    }

    try:
        image_path = (
            PROJECT_ROOT / record["standardized_path"]
        )

        with Image.open(image_path) as source:
            image_bgr = np.asarray(
                source.convert("RGB")
            )[:, :, ::-1].copy()

        faces = face_detector.get(image_bgr)

        details = [
            {
                "face_index": index,
                "bbox": face.bbox.tolist(),
                "landmarks": (
                    face.kps.tolist()
                    if face.kps is not None
                    else None
                ),
                "det_score": float(face.det_score),
            }
            for index, face in enumerate(faces)
        ]

        n_faces = len(details)

        if n_faces == 0:
            face_status = "no_face"
        elif n_faces == 1:
            face_status = "one_face"
        else:
            face_status = "multiple_faces"

        result.update({
            "n_faces": n_faces,
            "face_status": face_status,
            "faces_json": json.dumps(details),
        })

    except Exception as exc:
        result["detection_error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    result["detection_seconds"] = round(
        time.perf_counter() - started, 3
    )

    audit_live_df = pd.concat(
        [audit_live_df, pd.DataFrame([result])],
        ignore_index=True
    )

    # Tulis file sementara, lalu ganti CSV utama setelah selesai.
    temporary_path = AUDIT_PATH.with_suffix(".tmp")

    audit_live_df.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig"
    )

    temporary_path.replace(AUDIT_PATH)

print("Total foto tercatat:", len(audit_live_df))

## 15. Review manual: checkpoint interaktif

Foto dengan banyak wajah harus dipilih berdasarkan pemilik label, bukan otomatis wajah terbesar. Bagian ini sengaja memerlukan interaksi lokal; notebook tidak menjanjikan Run All tanpa jeda.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

from PIL import Image, ImageDraw
from IPython.display import display

from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"

AUDIT_PATH = METADATA_DIR / "face_detection_audit.csv"
SELECTION_PATH = METADATA_DIR / "manual_face_selection.csv"

if not AUDIT_PATH.is_file():
    raise FileNotFoundError(
        f"Hasil deteksi tidak ditemukan: {AUDIT_PATH}"
    )

face_audit_df = pd.read_csv(AUDIT_PATH)

review_df = (
    face_audit_df.loc[
        face_audit_df["face_status"].eq("multiple_faces")
    ]
    .reset_index(drop=True)
)

print("Foto dalam hasil deteksi:", len(face_audit_df))
print("Foto untuk review       :", len(review_df))
print("Pilihan sebelumnya ada  :", SELECTION_PATH.exists())
if not SELECTION_PATH.exists():
    pd.DataFrame(columns=[
        "image_id", "person_id", "person_name", "selected_face_index",
        "selected_bbox_json", "selected_landmarks_json", "reviewed_at_utc",
    ]).to_csv(SELECTION_PATH, index=False, encoding="utf-8-sig")

In [ ]:
import json
import pandas as pd

from datetime import datetime, timezone
from PIL import Image, ImageDraw
from IPython.display import display

CURRENT_REVIEW = None


def review_next():
    if not SHOW_PRIVATE_PREVIEWS:
        raise ValueError("Aktifkan SHOW_PRIVATE_PREVIEWS untuk melihat foto sebelum memilih wajah.")
    global CURRENT_REVIEW
    CURRENT_REVIEW = None

    if SELECTION_PATH.exists():
        selections = pd.read_csv(SELECTION_PATH)
        completed_ids = set(selections["image_id"])
    else:
        completed_ids = set()

    pending = review_df.loc[
        ~review_df["image_id"].isin(completed_ids)
    ]

    if pending.empty:
        print(f"Seluruh {len(review_df)} foto sudah direview.")
        return

    record = pending.iloc[0].to_dict()
    faces = json.loads(record["faces_json"])

    with Image.open(
        PROJECT_ROOT / record["standardized_path"]
    ) as source:
        original = source.convert("RGB")

    preview = original.copy()
    preview.thumbnail((1000, 800))

    sx = preview.width / original.width
    sy = preview.height / original.height

    draw = ImageDraw.Draw(preview)

    for face in faces:
        x1, y1, x2, y2 = face["bbox"]

        box = (
            x1 * sx, y1 * sy,
            x2 * sx, y2 * sy
        )

        draw.rectangle(box, outline="red", width=3)

        label_x = max(0, int(box[0]))
        label_y = max(0, int(box[1]) - 22)

        draw.rectangle(
            (
                label_x, label_y,
                label_x + 85, label_y + 22
            ),
            fill="white"
        )

        draw.text(
            (label_x + 4, label_y + 4),
            f"face {face['face_index']}",
            fill="black"
        )

    completed = review_df["image_id"].isin(
        completed_ids
    ).sum()

    print(f"Sudah direview: {completed}/{len(review_df)}")
    print("Pemilik folder:", record["person_name"])
    print("Image ID      :", record["image_id"])

    display(preview)

    CURRENT_REVIEW = record

    print(
        "Lihat gambarnya, lalu simpan nomor wajah "
        "menggunakan pilih_wajah(nomor)."
    )


def pilih_wajah(face_index):
    global CURRENT_REVIEW

    if CURRENT_REVIEW is None:
        raise ValueError(
            "Jalankan review_next() untuk menampilkan foto dahulu."
        )

    record = CURRENT_REVIEW
    faces = json.loads(record["faces_json"])

    selected = next(
        (
            face for face in faces
            if face["face_index"] == face_index
        ),
        None
    )

    if selected is None:
        valid_indices = [
            face["face_index"] for face in faces
        ]
        raise ValueError(
            f"Nomor wajah harus salah satu dari {valid_indices}."
        )

    new_selection = {
        "image_id": record["image_id"],
        "person_id": record["person_id"],
        "person_name": record["person_name"],
        "selected_face_index": face_index,
        "selected_bbox_json": json.dumps(selected["bbox"]),
        "selected_landmarks_json": json.dumps(
            selected["landmarks"]
        ),
        "reviewed_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    if SELECTION_PATH.exists():
        selections = pd.read_csv(SELECTION_PATH)
        selections = selections.loc[
            selections["image_id"].ne(record["image_id"])
        ]

        selections = pd.concat(
            [selections, pd.DataFrame([new_selection])],
            ignore_index=True
        )
    else:
        selections = pd.DataFrame([new_selection])

    temporary_path = SELECTION_PATH.with_suffix(".tmp")

    selections.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig"
    )

    temporary_path.replace(SELECTION_PATH)

    print("Tersimpan:", record["image_id"])
    print("Wajah dipilih: face", face_index)

    CURRENT_REVIEW = None

Aktifkan pratinjau hanya di komputer lokal. Jalankan `review_next()`, lihat gambar, lalu panggil `pilih_wajah(nomor)` sesuai label yang terlihat. Ulangi sampai seluruh foto selesai. Tidak ada pilihan wajah bawaan yang dijalankan otomatis.

In [ ]:
# Jalankan secara interaktif bila ada foto dengan beberapa wajah:
# SHOW_PRIVATE_PREVIEWS = True
# review_next()
# pilih_wajah(1)  # Ganti sesuai label yang benar-benar terlihat.
# review_next()

In [ ]:
import pandas as pd

audit_df = pd.read_csv(
    METADATA_DIR / "face_detection_audit.csv"
)

review_df = (
    audit_df.loc[
        audit_df["face_status"].eq("multiple_faces")
    ]
    .reset_index(drop=True)
)

selection_path = METADATA_DIR / "manual_face_selection.csv"

if selection_path.exists():
    selections_df = pd.read_csv(selection_path)
    reviewed_ids = set(selections_df["image_id"])
else:
    reviewed_ids = set()

pending_df = review_df.loc[
    ~review_df["image_id"].isin(reviewed_ids)
]

completed = len(review_df) - len(pending_df)

print("Total foto perlu review:", len(review_df))
print("Sudah direview         :", completed)
print("Belum direview         :", len(pending_df))

if pending_df.empty:
    print("\nSELESAI: seluruh foto yang memerlukan review sudah dipilih.")
else:
    print("\nFoto yang belum direview:")
    display(
        pending_df[["image_id", "person_name"]]
    )

## 16. Validasi pilihan wajah

Notebook berhenti di sini jika masih ada wajah yang belum dipilih, tidak terdeteksi, atau gagal diproses.

In [ ]:
import json
import numpy as np
import pandas as pd

audit_df = pd.read_csv(
    METADATA_DIR / "face_detection_audit.csv"
)

manual_df = pd.read_csv(
    METADATA_DIR / "manual_face_selection.csv"
)

expected_images = int(pd.read_csv(
    METADATA_DIR / "standardization_manifest.csv"
)["status"].eq("success").sum())
if len(audit_df) != expected_images:
    raise ValueError("Jumlah hasil deteksi tidak sesuai manifest standardisasi.")

if audit_df["image_id"].duplicated().any():
    raise ValueError("Ada image_id duplikat pada hasil deteksi.")

if manual_df["image_id"].duplicated().any():
    raise ValueError("Ada pilihan manual duplikat.")

manual_map = manual_df.set_index("image_id").to_dict("index")

selected_rows = []
problems = []

for record in audit_df.to_dict("records"):
    try:
        faces = json.loads(record["faces_json"])

        if record["face_status"] == "one_face" and len(faces) == 1:
            selected = faces[0]
            method = "single_face"

        elif record["face_status"] == "multiple_faces":
            choice = manual_map.get(record["image_id"])

            if choice is None:
                raise ValueError("Pilihan manual belum tersedia.")

            selected = next(
                face for face in faces
                if face["face_index"]
                == int(choice["selected_face_index"])
            )

            # Periksa apakah pilihan berasal dari deteksi yang sama.
            saved_bbox = np.asarray(
                json.loads(choice["selected_bbox_json"])
            )
            current_bbox = np.asarray(selected["bbox"])

            if (
                saved_bbox.shape != current_bbox.shape
                or not np.allclose(
                    saved_bbox, current_bbox,
                    atol=0.001, rtol=0
                )
            ):
                raise ValueError("Pilihan perlu direview ulang.")

            method = "manual"

        else:
            raise ValueError("Status deteksi belum siap.")

        landmarks = np.asarray(
            selected["landmarks"],
            dtype=np.float32
        )

        if (
            landmarks.shape != (5, 2)
            or not np.isfinite(landmarks).all()
        ):
            raise ValueError("Landmark wajah tidak valid.")

        selected_rows.append({
            **record,
            "selected_face_index": selected["face_index"],
            "selected_bbox_json": json.dumps(selected["bbox"]),
            "selected_landmarks_json": json.dumps(
                selected["landmarks"]
            ),
            "selection_method": method,
        })

    except Exception as exc:
        problems.append({
            "image_id": record["image_id"],
            "person_name": record["person_name"],
            "reason": str(exc) or type(exc).__name__,
        })

print("Siap crop :", len(selected_rows))
print("Bermasalah:", len(problems))

if problems:
    display(pd.DataFrame(problems))
    raise ValueError("Selesaikan masalah sebelum menjalankan crop.")

selected_faces_df = pd.DataFrame(selected_rows)

selected_faces_df.to_csv(
    METADATA_DIR / "face_selection_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

## 17. Crop dan alignment 112 × 112

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm
from insightface.utils import face_align


from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"

PROCESSED_DIR = (
    PROJECT_ROOT / "dataset" / "processed" / "arcface_112"
)

selection_path = METADATA_DIR / "face_selection_manifest.csv"
alignment_path = METADATA_DIR / "alignment_manifest.csv"

# Baca pilihan wajah yang sudah disiapkan.
selected_faces_df = pd.read_csv(selection_path)

expected_images = int(pd.read_csv(
    METADATA_DIR / "standardization_manifest.csv"
)["status"].eq("success").sum())
if len(selected_faces_df) != expected_images:
    raise ValueError(
        f"Manifest pilihan berisi {len(selected_faces_df)} foto. "
        "Jalankan kembali pemeriksaan sebelum crop hingga seluruh foto siap."
    )

if selected_faces_df["image_id"].duplicated().any():
    raise ValueError("Ada image_id duplikat pada manifest pilihan.")

missing_files = [
    path
    for path in selected_faces_df["standardized_path"]
    if not (PROJECT_ROOT / path).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Ada foto sumber yang tidak ditemukan: {missing_files[:3]}"
    )

print("Foto siap diproses:", len(selected_faces_df), flush=True)
print("Folder hasil      :", PROCESSED_DIR, flush=True)

alignment_rows = []
started = time.perf_counter()

records = selected_faces_df.to_dict("records")

for record in tqdm(
    records,
    desc="Crop dan alignment",
    unit="foto"
):
    output_dir = PROCESSED_DIR / record["person_id"]
    output_path = output_dir / f"{record['image_id']}.png"

    result = {
        **record,
        "aligned_path": str(output_path.relative_to(PROJECT_ROOT)),
        "alignment_status": "failed",
        "alignment_error": None,
    }

    try:
        source_path = (
            PROJECT_ROOT / record["standardized_path"]
        )

        with Image.open(source_path) as source:
            image_bgr = np.asarray(
                source.convert("RGB")
            )[:, :, ::-1].copy()

        landmarks = np.asarray(
            json.loads(record["selected_landmarks_json"]),
            dtype=np.float32
        )

        if (
            landmarks.shape != (5, 2)
            or not np.isfinite(landmarks).all()
        ):
            raise ValueError("Landmark wajah tidak valid.")

        # Sejajarkan wajah menggunakan lima landmark.
        aligned_bgr = face_align.norm_crop(
            image_bgr,
            landmark=landmarks,
            image_size=112,
            mode="arcface"
        )

        if aligned_bgr.shape != (112, 112, 3):
            raise ValueError("Ukuran hasil alignment tidak sesuai.")

        output_dir.mkdir(parents=True, exist_ok=True)

        temporary_image = output_path.with_suffix(".tmp")

        Image.fromarray(
            aligned_bgr[:, :, ::-1].copy()
        ).save(temporary_image, format="PNG")

        # Periksa hasil sebelum dijadikan file utama.
        with Image.open(temporary_image) as saved:
            saved.load()

            if saved.size != (112, 112) or saved.mode != "RGB":
                raise ValueError("File hasil crop tidak sesuai.")

        temporary_image.replace(output_path)

        result["alignment_status"] = "success"

    except Exception as exc:
        result["alignment_error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    alignment_rows.append(result)

    # Simpan laporan setiap selesai satu foto.
    temporary_manifest = alignment_path.with_suffix(".tmp")

    pd.DataFrame(alignment_rows).to_csv(
        temporary_manifest,
        index=False,
        encoding="utf-8-sig"
    )

    temporary_manifest.replace(alignment_path)


# Ringkasan hasil.
alignment_df = pd.read_csv(alignment_path)

success_df = alignment_df.loc[
    alignment_df["alignment_status"].eq("success")
]

failed_df = alignment_df.loc[
    alignment_df["alignment_status"].eq("failed")
]

print(
    f"\nSelesai dalam {time.perf_counter() - started:.1f} detik."
)
print("Total diproses    :", len(alignment_df))
print("Berhasil          :", len(success_df))
print("Gagal             :", len(failed_df))
print("Identitas berhasil:", success_df["person_id"].nunique())
print("Manifest tersimpan:", alignment_path)

display(
    failed_df[
        ["image_id", "person_name", "alignment_error"]
    ]
)
if len(failed_df):
    raise ValueError("Perbaiki kegagalan alignment sebelum notebook 02.")

## 18. Inspeksi visual lokal (opsional)

In [ ]:
import math
import textwrap
import pandas as pd

from PIL import Image, ImageDraw
from IPython.display import display

alignment_df = pd.read_csv(
    METADATA_DIR / "alignment_manifest.csv"
)

aligned_ok_df = (
    alignment_df.loc[
        alignment_df["alignment_status"].eq("success")
    ]
    .sort_values(["person_id", "image_id"])
    .reset_index(drop=True)
)

person_ids = aligned_ok_df["person_id"].unique().tolist()


def show_alignment_page(page=0, people_per_page=8):
    total_pages = math.ceil(
        len(person_ids) / people_per_page
    )

    if not 0 <= page < total_pages:
        raise ValueError(
            f"Nomor halaman harus 0 sampai {total_pages - 1}."
        )

    selected_ids = person_ids[
        page * people_per_page:
        (page + 1) * people_per_page
    ]

    max_photos = int(
        aligned_ok_df.groupby("person_id").size().max()
    )

    label_width = 210
    tile_width = 145
    row_height = 165

    sheet = Image.new(
        "RGB",
        (
            label_width + max_photos * tile_width,
            len(selected_ids) * row_height
        ),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for row_index, person_id in enumerate(selected_ids):
        group = aligned_ok_df.loc[
            aligned_ok_df["person_id"].eq(person_id)
        ]

        top = row_index * row_height
        person_name = group.iloc[0]["person_name"]

        label = (
            person_id + "\n"
            + "\n".join(
                textwrap.wrap(str(person_name), width=28)
            )
        )

        draw.multiline_text(
            (8, top + 15),
            label,
            fill="black",
            spacing=5
        )

        for column, record in enumerate(
            group.to_dict("records")
        ):
            left = label_width + column * tile_width
            image_path = PROJECT_ROOT / record["aligned_path"]

            with Image.open(image_path) as source:
                face = source.convert("RGB")

            # Tampilkan ukuran asli 112 x 112.
            sheet.paste(face, (left + 10, top + 8))

            image_code = record["image_id"].split("_img_")[-1]

            draw.text(
                (left + 8, top + 126),
                f"img {column + 1}",
                fill="black"
            )

            draw.text(
                (left + 8, top + 143),
                image_code,
                fill="black"
            )

    print(f"Halaman {page + 1}/{total_pages}")
    print("Kode di bawah wajah adalah bagian akhir image_id.")

    display(sheet)

In [ ]:
# SHOW_PRIVATE_PREVIEWS = True
if SHOW_PRIVATE_PREVIEWS:
    show_alignment_page(0)